# 03 - Mainnet CLMM

Concentrated-liquidity AMM analysis. This notebook is intentionally separate from CPMM because CLMM needs tick/active-liquidity state and cannot reuse the CPMM closed-form baseline.

Current inputs:
- `results/historical_clmm_decoded.csv` — decoded historical CLMM swap observations
- `results/historical_clmm_swaps_status.csv` — inclusion/exclusion ledger
- `results/historical_clmm_pipeline_summary.csv` — collection/decode funnel
- `results/historical_clmm_live_swaps.csv` — live-discovered CLMM swaps for short archive windows
- `results/historical_clmm_live_snapshots.csv` — repeated snapshots of live watchlisted CLMM accounts
- `results/historical_clmm_live_candidates.csv` — readiness filter over live decoded swaps

Attack-evaluation input:
- `results/historical_clmm_candidates.csv` — CLMM attack evaluation rows; rejected rows remain useful as methodology/status evidence


## Load data


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    best_conditions,
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_counterfactual_summary,
    hypothesis_scorecard,
    load_inputs,
    plot_realized_heatmap,
    plot_sensitivity_lines,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

clmm = inputs["frames"]["historical_clmm"]
clmm_decoded = inputs["frames"]["historical_clmm_decoded"]
clmm_status = inputs["frames"]["historical_clmm_swaps_status"]
clmm_summary = inputs["frames"]["historical_clmm_pipeline_summary"]
clmm_state_probe = inputs["frames"]["historical_clmm_state_probe"]
clmm_live_swaps = inputs["frames"]["historical_clmm_live_swaps"]
clmm_live_snapshots = inputs["frames"]["historical_clmm_live_snapshots"]
clmm_live_candidates = inputs["frames"]["historical_clmm_live_candidates"]
cpmm = inputs["frames"]["historical_cpmm"]

display(data_readiness(ROOT, inputs, [
    "historical_clmm_decoded",
    "historical_clmm_swaps_status",
    "historical_clmm_pipeline_summary",
    "historical_clmm_state_probe",
    "historical_clmm_live_swaps",
    "historical_clmm_live_snapshots",
    "historical_clmm_live_candidates",
    "historical_clmm",
    "historical_cpmm",
]))


## Scope check


In [ ]:
if not clmm.empty:
    if "model_status" in clmm:
        display(clmm["model_status"].value_counts().rename_axis("model_status").reset_index(name="rows"))
    if "rejection_reason" in clmm:
        display(clmm["rejection_reason"].fillna("evaluated").value_counts().rename_axis("rejection_reason").reset_index(name="rows"))
    display(historical_counterfactual_summary(clmm))
elif not clmm_decoded.empty:
    display(Markdown(
        f"Decoded `{len(clmm_decoded)}` CLMM swap observation(s), but no counterfactual candidate CSV exists yet. "
        "This is expected until historical PoolState/tick-array pre-state and CLMM replay validation are implemented."
    ))
    display(clmm_summary)
    if not clmm_state_probe.empty:
        display(clmm_state_probe[[
            "pool_label",
            "slot",
            "instruction_index",
            "required_account_count",
            "current_probe_slot",
            "current_pool_state_ok",
            "current_amm_config_ok",
            "current_tick_arrays_ok",
            "historical_state_available",
            "candidate_ready",
        ]].head(20))
    display(
        clmm_decoded[[
            "pool_label",
            "slot",
            "signature",
            "instruction_index",
            "swap_variant",
            "direction",
            "amount_in",
            "actual_amount_out",
            "historical_state_status",
        ]].head(20)
    )
else:
    display(Markdown(
        "No decoded CLMM observations exist yet. Run `cargo run -p fork --bin historical_clmm -- --pool clmm_wsol_usdc --limit-per-pool 50 run-all`."
    ))
    if not clmm_status.empty:
        display(clmm_status["analysis_status"].value_counts().rename_axis("analysis_status").reset_index(name="rows"))

if not clmm_live_swaps.empty:
    display(Markdown(
        f"Live collector observed `{len(clmm_live_swaps)}` signature row(s). "
        "Rows become useful for candidate construction only when all required accounts were known before discovery and a previous snapshot exists."
    ))
    display(clmm_live_swaps["status"].value_counts().rename_axis("status").reset_index(name="rows"))
    if not clmm_live_snapshots.empty:
        display(clmm_live_snapshots["account_role"].value_counts().rename_axis("account_role").reset_index(name="snapshots"))

if not clmm_live_candidates.empty:
    ready = clmm_live_candidates["live_candidate_ready"].astype(str).str.lower().isin(["true", "1"])
    display(Markdown(f"Live CLMM readiness rows: `{len(clmm_live_candidates)}`, ready: `{int(ready.sum())}`."))
    if "rejection_reason" in clmm_live_candidates:
        display(clmm_live_candidates.loc[~ready, "rejection_reason"].value_counts().rename_axis("rejection_reason").reset_index(name="rows"))


## Required CLMM candidate fields


In [ ]:
required = pd.DataFrame([
    {"field": "pool_type", "why": "distinguish raydium_clmm/orca_whirlpool from cpmm"},
    {"field": "pool_label, pool_address", "why": "group results by selected pool"},
    {"field": "slot, signature, instruction_index", "why": "trace every counterfactual row back to a historical swap"},
    {"field": "amount_in, min_amount_out, actual_amount_out", "why": "victim size and slippage bound"},
    {"field": "sqrt_price_x64_before, liquidity_before, tick_current_before", "why": "CLMM state at victim pre-state"},
    {"field": "tick_arrays_before", "why": "needed for executable CLMM swap simulation across ticks"},
    {"field": "fee_rate, protocol_fee_rate", "why": "fee-aware profitability"},
    {"field": "tx_cost_per_leg", "why": "net profit, not just gross extraction"},
])
display(required)


## CPMM vs CLMM comparison


In [ ]:
if clmm.empty or cpmm.empty:
    display(Markdown("Counterfactual CPMM vs CLMM comparison waits for both historical CPMM and CLMM candidate outputs."))
    if not clmm_decoded.empty:
        display(Markdown("CLMM decode coverage is available, but it is not profitability evidence yet."))
else:
    cpmm_summary = historical_counterfactual_summary(cpmm).assign(family="CPMM")
    clmm_candidate_summary = historical_counterfactual_summary(clmm).assign(family="CLMM")
    display(pd.concat([cpmm_summary, clmm_candidate_summary], ignore_index=True))


## Replay validation

`replay_error_bps = |fair_amount_out - actual_amount_out| / actual_amount_out
* 10000`. The pipeline rejects rows above the configured tolerance
(`--replay-tolerance-bps`, default 100). Stats below are over the
`evaluated` rows that passed the tolerance, plus a count of rejected
outliers. See `.docs/audits/2026-05-clmm-model-validation.md`.


In [ ]:
from helpers.clmm_attacks import replay_validation_summary

if clmm.empty or "replay_error_bps" not in clmm.columns:
    display(Markdown("No `replay_error_bps` column — re-run `evaluate-live-attacks`."))
else:
    stats = replay_validation_summary(clmm)
    display(pd.DataFrame([stats]))
    err = pd.to_numeric(
        clmm.loc[clmm["model_status"].astype(str).eq("evaluated"), "replay_error_bps"],
        errors="coerce",
    ).dropna()
    if not err.empty:
        ax = err.plot(kind="hist", bins=20, edgecolor="black")
        ax.set_xlabel("replay_error_bps (evaluated rows)")
        ax.set_ylabel("swaps")
        ax.set_title("Replay error: model fair_amount_out vs on-chain actual")
    if (clmm.get("rejection_reason", "") == "victim_replay_mismatch").any():
        outliers = clmm[clmm["rejection_reason"] == "victim_replay_mismatch"][
            ["pool_label", "signature", "direction", "amount_in", "actual_amount_out", "fair_amount_out", "replay_error_bps"]
        ]
        display(Markdown("### Replay outliers rejected by the pipeline"))
        display(outliers)


## Best-attempt diagnostics

Why does the live CLMM evaluator return `0` profitable rows on this sample?
The grid search now keeps the highest-`net_profit` attempt regardless of
profitability, so the `best_attempt_*` columns expose the failure mode for
every evaluated candidate (fee drag, victim slippage breach, victim too
small). `rejection_reason` distinguishes `no_grid_result`,
`best_attempt_infeasible`, and `best_attempt_unprofitable`.


In [ ]:
from helpers.clmm_attacks import (
    attach_diagnostics,
    rejection_breakdown,
    best_attempt_summary,
    fee_drag_vs_gross,
)

if clmm.empty or "best_attempt_net_profit" not in clmm.columns:
    display(Markdown(
        "No `best_attempt_*` columns in `historical_clmm_candidates.csv`. "
        "Re-run `evaluate-live-attacks` with the current `attack.rs` to populate them."
    ))
else:
    diag = attach_diagnostics(clmm)
    display(Markdown(
        f"Evaluated rows: `{int(diag['evaluated'].sum())}` of `{len(diag)}`. "
        f"Profitable: `{int(diag.get('attack_profitable', False).astype(bool).sum())}`."
    ))
    display(rejection_breakdown(diag))
    display(best_attempt_summary(diag))
    drag_series = fee_drag_vs_gross(diag)
    if not drag_series.empty:
        display(Markdown(
            f"`best_attempt_gross_profit - 2*tx_cost_per_leg` over evaluated rows: "
            f"min `{drag_series.min():,.0f}`, median `{drag_series.median():,.0f}`, "
            f"max `{drag_series.max():,.0f}`, share <= 0: "
            f"`{(drag_series <= 0).mean():.0%}`."
        ))


## Profitability landscape

Outcome buckets per pool, with the capital-efficiency and slippage-breach
caveats called out for the profitable subset. See
`.docs/audits/2026-05-clmm-optimizer-port.md` for the discussion of why
the headline profitable share should not be quoted without these filters.


In [ ]:
from helpers.clmm_attacks import (
    plot_attempt_buckets_by_pool,
    plot_frontrun_vs_victim,
    plot_victim_slippage_breach,
)

if not clmm.empty and "best_attempt_net_profit" in clmm.columns:
    fig = plot_attempt_buckets_by_pool(clmm)
    if fig is not None:
        plt.show()
    fig = plot_frontrun_vs_victim(clmm)
    if fig is not None:
        plt.show()
    fig = plot_victim_slippage_breach(clmm)
    if fig is not None:
        plt.show()


## Thesis-ready takeaways


In [ ]:
if clmm.empty:
    lines = [
        "- CLMM profitability remains an empirical gap, not a result.",
        f"- Decoded CLMM observations available: `{len(clmm_decoded)}` rows.",
        f"- CLMM state probe rows available: `{len(clmm_state_probe)}` rows.",
        "- Next blocker: historical PoolState/tick-array pre-state and victim-swap replay validation.",
    ]
    display(Markdown("\n".join(lines)))
else:
    display(Markdown(f"- Historical CLMM counterfactual candidates loaded: `{len(clmm)}` rows."))
